In [2]:
#imports
import polars as pl
import json

import requests
from deepface import DeepFace
from tqdm import tqdm


In [28]:
path=r'C:\Users\Porter\Desktop\jcp\merge_data\jcpst-sessions-2026-04-21-17-27-55.json'

In [33]:
import json

with open(path, "r", encoding="utf-8") as f:
    obj = json.load(f)

sessions = pl.DataFrame(obj["sessions"])



In [31]:
def expand_struct(df, colname):
    return df.with_columns(
        pl.col(colname).struct.unnest()
    )


#for treatment cols
def expand_list_struct_drop_inner(df: pl.DataFrame, col: str) -> pl.DataFrame:
    exploded = df.explode(col)

    inner_names = [f.name for f in exploded.schema[col].fields]
    outer_names = set(exploded.columns) - {col}

    keep_inner = [name for name in inner_names if name not in outer_names]

    return (
        exploded
        .with_columns(
            pl.struct(
                [pl.col(col).struct.field(name).alias(name) for name in keep_inner]
            ).alias(col)
        )
        .unnest(col)
    )


# def expand_list_struct_drop_inner(df: pl.DataFrame, col: str) -> pl.DataFrame:
#     exploded = df.explode(col)

#     inner_names = [f.name for f in exploded.schema[col].fields]
#     outer_names = set(exploded.columns) - {col}

#     keep_inner = [name for name in inner_names if name not in outer_names]

#     return (
#         exploded
#         .with_columns(
#             pl.struct(
#                 [pl.col(col).struct.field(name).alias(name) for name in keep_inner]
#             ).alias(col)
#         )
#         .unnest(col)
#     )

In [34]:
sessions=expand_struct(sessions,'session')

In [35]:
sessions=expand_list_struct_drop_inner(sessions,'linkedin_rows')


In [36]:
sessions=expand_struct(sessions,'profile_data')

In [37]:
sessions=expand_list_struct_drop_inner(sessions,'job_survey_rows')

In [40]:
sessions=sessions.select(pl.exclude(['session','profile_data']))

In [42]:
sessions=sessions.with_columns((pl.col("session_id") + "-" + pl.col("survey_id").fill_null('')).alias("session_survey_id"))

    # assert if it is a unique key
assert sessions.select(pl.col("session_survey_id").is_unique().all()).item()


In [43]:
sessions

pageviews,ip,date,survey_id,treatment_group,post_url,job_ad_url,survey_url,likely_apply,likely_accept,perform_job,frac_quals,hiring_manager,likely_interview,exper,other_skills,educ,race,gender,empstat,resume,submit_1,company_lenient,waste_time,stretch_roles,apply_with_most,required_less,remove_data,id,linkedin_member_id,email,verification_data,token_expires_at,last_synced_at,session_id,user_id,user_display_name,session_start,last_activity,session_end,duration_seconds,total_pageviews,visited_pages,treatment_snapshot,first_referrer,first_ip,last_ip,ip_hash,user_agent,device_summary,is_logged_in,login_state_changed,created_at,updated_at,sub,email_verified,name,locale,given_name,family_name,picture,session_survey_id
list[struct[15]],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,list[null],str,str,str,str,str,str,str,str,i64,i64,list[struct[2]],list[struct[7]],str,str,str,str,str,str,bool,bool,str,str,str,bool,str,struct[2],str,str,str,str
[],null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""f7fa559b9f7a2c799c18ad9fdd9870…",null,"""""","""2026-04-21 17:27:09""","""2026-04-21 17:27:09""",null,46,0,[],[],"""""","""192.0.84.117""","""192.0.84.117""","""55b50e648d132e75de60b15ae9e108…","""jetmon/1.0 (Jetpack Site Uptim…","""Desktop / Unknown Browser / Un…",false,false,"""2026-04-21 17:27:09""","""2026-04-21 17:27:09""",null,null,null,null,null,null,null,"""f7fa559b9f7a2c799c18ad9fdd9870…"
[],null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""3f26cb091ef56285e731e139adfff1…",null,"""""","""2026-04-21 17:22:09""","""2026-04-21 17:22:09""",null,346,0,[],[],"""""","""192.0.84.117""","""192.0.84.117""","""55b50e648d132e75de60b15ae9e108…","""jetmon/1.0 (Jetpack Site Uptim…","""Desktop / Unknown Browser / Un…",false,false,"""2026-04-21 17:22:09""","""2026-04-21 17:22:09""",null,null,null,null,null,null,null,"""3f26cb091ef56285e731e139adfff1…"
[],null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""a590626c17454f9d94ff3cc4ce1d22…",null,"""""","""2026-04-21 17:17:09""","""2026-04-21 17:17:09""",null,646,0,[],[],"""""","""192.0.84.117""","""192.0.84.117""","""55b50e648d132e75de60b15ae9e108…","""jetmon/1.0 (Jetpack Site Uptim…","""Desktop / Unknown Browser / Un…",false,false,"""2026-04-21 17:17:09""","""2026-04-21 17:17:09""",null,null,null,null,null,null,null,"""a590626c17454f9d94ff3cc4ce1d22…"
[],null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""ef9404ed9cfa58957d5744c3d32ac4…",null,"""""","""2026-04-21 17:12:09""","""2026-04-21 17:12:09""",null,946,0,[],[],"""""","""192.0.84.117""","""192.0.84.117""","""55b50e648d132e75de60b15ae9e108…","""jetmon/1.0 (Jetpack Site Uptim…","""Desktop / Unknown Browser / Un…",false,false,"""2026-04-21 17:12:09""","""2026-04-21 17:12:09""",null,null,null,null,null,null,null,"""ef9404ed9cfa58957d5744c3d32ac4…"
[],null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""66f836eb11d4d192e072fc5ef0602e…",null,"""""","""2026-04-21 17:07:10""","""2026-04-21 17:07:10""",null,1245,0,[],[],"""""","""192.0.84.117""","""192.0.84.117""","""55b50e648d132e75de60b15ae9e108…","""jetmon/1.0 (Jetpack Site Uptim…","""Desktop / Unknown Browser / Un…",false,false,"""2026-04-21 17:07:10""","""2026-04-21 17:07:10""",null,null,null,null,null,null,null,"""66f836eb11d4d192e072fc5ef0602e…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

In [44]:
signed_in=sessions.filter(pl.col('user_id').is_not_null())
signed_in

pageviews,ip,date,survey_id,treatment_group,post_url,job_ad_url,survey_url,likely_apply,likely_accept,perform_job,frac_quals,hiring_manager,likely_interview,exper,other_skills,educ,race,gender,empstat,resume,submit_1,company_lenient,waste_time,stretch_roles,apply_with_most,required_less,remove_data,id,linkedin_member_id,email,verification_data,token_expires_at,last_synced_at,session_id,user_id,user_display_name,session_start,last_activity,session_end,duration_seconds,total_pageviews,visited_pages,treatment_snapshot,first_referrer,first_ip,last_ip,ip_hash,user_agent,device_summary,is_logged_in,login_state_changed,created_at,updated_at,sub,email_verified,name,locale,given_name,family_name,picture,session_survey_id
list[struct[15]],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,list[null],str,str,str,str,str,str,str,str,i64,i64,list[struct[2]],list[struct[7]],str,str,str,str,str,str,bool,bool,str,str,str,bool,str,struct[2],str,str,str,str
"[{2038,""1295f2d4cfb3d6ded8462211ed26e674a32cb0e96ff4e563658ab2e5df9a4e81"",6,""2026-04-21 16:48:47"",""https://jobconnectionsproject.org/jobs/"",""/jobs/"","""",""https://jobconnectionsproject.org/"",""128.187.116.32"",""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"",622,""Search for Jobs"",false,false,true}, {2039,""1295f2d4cfb3d6ded8462211ed26e674a32cb0e96ff4e563658ab2e5df9a4e81"",6,""2026-04-21 16:48:51"",""https://jobconnectionsproject.org/jobs/corporate-executive-pastry-chef-troy-mi-us/"",""/jobs/corporate-executive-pastry-chef-troy-mi-us/"","""",""https://jobconnectionsproject.org/jobs/"",""128.187.116.32"",""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"",3590,""Corporate Executive Pastry Chef &#8211; Troy, MI, US"",false,false,true}, … {2047,""1295f2d4cfb3d6ded8462211ed26e674a32cb0e96ff4e563658ab2e5df9a4e81"",6,""2026-04-21 17:26:12"",""https://jobconnectionsproject.org/"",""/"","""",""https://jobconnectionsproject.org/survey/"",""128.187.116.32"",""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"",67,""Home"",false,false,true}]","""128.187.116.32""","""2026-04-21 04:49:16""","""8680426331""","""0""","""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""Unlikely""","""Equally Likely and Unlikely""","""""","""""","""may or may not""","""""","""""","""""","""Some college""","""Multiracial or Biracial""","""A gender identity not listed h…","""Employed (part-time)""","""""","""Apply now on company website""","""""","""3""","""""","""3""","""""","""""",1,"""AKLfR3WKvk""","""porter.olson11@gmail.com""",[],"""2026-06-08 03:54:09""","""2026-04-09 03:54:10""","""1295f2d4cfb3d6ded8462211ed26e6…","""6""","""Porter Olson""","""2026-04-21 16:48:47""","""2026-04-21 17:26:13""",null,2348,10,"[{""2026-04-21 16:48:47"",""https://jobconnectionsproject.org/jobs/""}, {""2026-04-21 16:48:51"",""https://jobconnectionsproject.org/jobs/corporate-executive-pastry-chef-troy-mi-us/""}, … {""2026-04-21 17:26:12"",""https://jobconnectionsproject.org/""}]","[{""128.187.116.32"",""26-04-21 04:48:52"",""8680426331"",0,""https://jobconnectionsproject.org/jobs/"",""https://jobconnectionsproject.org/jobs/corporate-executive-pastry-chef-troy-mi-us/"",""1295f2d4cfb3d6ded8462211ed26e674a32cb0e96ff4e563658ab2e5df9a4e81""}, {""128.187.116.32"",""26-04-21 05:25:54"",""3750118378"",0,""https://jobconnectionsproject.org/jobs/"",""https://jobconnectionsproject.org/jobs/baker-east-syracuse-ny-us/"",""1295f2d4cfb3d6ded8462211ed26e674a32cb0e96ff4e563658ab2e5df9a4e81""}]","""https://jobconnectionsproject.…","""128.187.116.32""","""128.187.116.32""","""a2479127ba3990c4c2859b95174800…","""Mozilla/5.0 (Windows NT 10.0; …","""Desktop / Chrome / Windows""",true,true,"""2026-04-21 16:48:47""","""2026-04-21 17:27:52"""

In [45]:
signed_in_detail=signed_in.select(['given_name','family_name','picture']).filter(pl.col('given_name').is_not_null())
signed_in_detail=signed_in_detail.unique()
signed_in_detail

picture_link_list=signed_in_detail['picture'].to_list()

In [47]:
rows = []

for link in tqdm(picture_link_list):
    row = {
        "picture": link,

        "df_age": None,
        "df_face_confidence": None,
        "df_dominant_gender": None,
        "df_dominant_race": None,

        "df_prob_male": None,
        "df_prob_female": None,

        "df_prob_white": None,
        "df_prob_black": None,
        "df_prob_asian": None,
        "df_prob_indian": None,
        "df_prob_middle_eastern": None,
        "df_prob_latino": None,

        "df_status": None,
        "df_error": None,
    }

    try:
        img_data = requests.get(link, timeout=20).content

        with open("temp.jpg", "wb") as f:
            f.write(img_data)

        result = DeepFace.analyze(
            img_path="temp.jpg",
            actions=["age", "gender", "race"],
            enforce_detection=True
        )

        face = result[0]

        row["df_age"] = face.get("age")
        row["df_face_confidence"] = face.get("face_confidence")
        row["df_dominant_gender"] = face.get("dominant_gender")
        row["df_dominant_race"] = face.get("dominant_race")

        gender_dict = face.get("gender", {})
        race_dict = face.get("race", {})

        row["df_prob_male"] = float(gender_dict["Man"]) if "Man" in gender_dict else None
        row["df_prob_female"] = float(gender_dict["Woman"]) if "Woman" in gender_dict else None

        row["df_prob_white"] = float(race_dict["white"]) if "white" in race_dict else None
        row["df_prob_black"] = float(race_dict["black"]) if "black" in race_dict else None
        row["df_prob_asian"] = float(race_dict["asian"]) if "asian" in race_dict else None
        row["df_prob_indian"] = float(race_dict["indian"]) if "indian" in race_dict else None
        row["df_prob_middle_eastern"] = float(race_dict["middle eastern"]) if "middle eastern" in race_dict else None
        row["df_prob_latino"] = float(race_dict["latino hispanic"]) if "latino hispanic" in race_dict else None

        row["df_status"] = "ok"

    except Exception as e:
        msg = str(e)

        if "Face could not be detected" in msg:
            row["df_status"] = "no_face_detected"
        else:
            row["df_status"] = "error"

        row["df_error"] = msg

    rows.append(row)

pic_df = pl.DataFrame(rows)

pic_df

100%|██████████| 4/4 [00:03<00:00,  1.20it/s]


picture,df_age,df_face_confidence,df_dominant_gender,df_dominant_race,df_prob_male,df_prob_female,df_prob_white,df_prob_black,df_prob_asian,df_prob_indian,df_prob_middle_eastern,df_prob_latino,df_status,df_error
str,i64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""https://media.licdn.com/dms/im…",25,0.92,"""Man""","""white""",99.571747,0.42825,94.859116,0.000324,0.029293,0.011143,2.819904,2.280221,"""ok""",null
"""https://media.licdn.com/dms/im…",23,0.92,"""Man""","""white""",98.778412,1.221596,68.907288,0.085356,1.472363,0.598139,13.370648,15.566208,"""ok""",null
"""https://media.licdn.com/dms/im…",32,0.9,"""Man""","""white""",99.723633,0.276374,98.395454,0.000001,0.000003,0.000235,1.036455,0.567853,"""ok""",null
"""https://media.licdn.com/dms/im…",null,null,null,null,null,null,null,null,null,null,null,null,"""no_face_detected""","""Face could not be detected in …"


In [50]:
import polars as pl
import pandas as pd
import gender_guesser.detector as gender
from ethnicolr import pred_fl_reg_name

d = gender.Detector(case_sensitive=False)

rows = []

for row in signed_in_detail.iter_rows(named=True):
    first_name = row["given_name"]
    last_name = row["family_name"]

    # gender-guesser
    gender_guess = d.get_gender(first_name) if first_name is not None else None

    temp_df = pd.DataFrame({
        "first_name": [first_name],
        "last_name": [last_name]
    })

    try:
        result = pred_fl_reg_name(temp_df, "last_name", "first_name")
        r = result.iloc[0]

        rows.append({
            "given_name": first_name,
            "family_name": last_name,

            # gender-guesser output
            "gg_gender_label": gender_guess,

            # ethnicolr output
            "eth_asian": r.get("asian"),
            "eth_hispanic": r.get("hispanic"),
            "eth_nh_black": r.get("nh_black"),
            "eth_nh_white": r.get("nh_white"),
            "eth_race_label": r.get("race"),
            "eth_processing_status": r.get("processing_status"),

            "ethnicolr_status": "ok"
        })

    except Exception as e:
        rows.append({
            "given_name": first_name,
            "family_name": last_name,

            "gg_gender_label": gender_guess,

            "eth_asian": None,
            "eth_hispanic": None,
            "eth_nh_black": None,
            "eth_nh_white": None,
            "eth_race_label": None,
            "eth_processing_status": None,

            "ethnicolr_status": f"error: {str(e)}"
        })

ethnicolr_gender_df = pl.DataFrame(rows)

ethnicolr_gender_df

2026-04-21 11:37:40,047 - INFO - Processing 1 full names
2026-04-21 11:37:40,087 - INFO - Applying Florida voter name model to 1 processable names (confidence interval: 1.0)
2026-04-21 11:37:40,089 - INFO - Data filtering summary: 1 → 1 rows (kept 100.0%)
2026-04-21 11:37:40,550 - INFO - Successfully predicted 1 of 1 names (100.0%)
2026-04-21 11:37:40,551 - INFO - Added columns: name_normalized_clean, nh_black, asian, __name, name_normalized, race, processing_status, nh_white, hispanic
2026-04-21 11:37:40,553 - INFO - Processing 1 full names
2026-04-21 11:37:40,556 - INFO - Applying Florida voter name model to 1 processable names (confidence interval: 1.0)
2026-04-21 11:37:40,558 - INFO - Data filtering summary: 1 → 1 rows (kept 100.0%)
2026-04-21 11:37:40,712 - INFO - Successfully predicted 1 of 1 names (100.0%)
2026-04-21 11:37:40,713 - INFO - Added columns: name_normalized_clean, nh_black, asian, __name, name_normalized, race, processing_status, nh_white, hispanic
2026-04-21 11:37:4

given_name,family_name,gg_gender_label,eth_asian,eth_hispanic,eth_nh_black,eth_nh_white,eth_race_label,eth_processing_status,ethnicolr_status
str,str,str,f64,f64,f64,f64,str,str,str
"""Porter""","""Olson""","""male""",0.000641,0.00363,0.007302,0.988427,"""nh_white""","""processed""","""ok"""
"""Spencer""","""Bailey""","""male""",0.003404,0.014741,0.130191,0.851664,"""nh_white""","""processed""","""ok"""
"""Tanner""","""Eastmond""","""male""",0.0021,0.012004,0.117149,0.868747,"""nh_white""","""processed""","""ok"""
"""Shark""","""Frisbee""","""unknown""",0.001475,0.005477,0.0429,0.950148,"""nh_white""","""processed""","""ok"""


In [48]:
sessions=sessions.join(pic_df, on=['picture'],how='left')

In [51]:
sessions=sessions.join(ethnicolr_gender_df,on=['given_name','family_name'])


In [52]:
sessions

pageviews,ip,date,survey_id,treatment_group,post_url,job_ad_url,survey_url,likely_apply,likely_accept,perform_job,frac_quals,hiring_manager,likely_interview,exper,other_skills,educ,race,gender,empstat,resume,submit_1,company_lenient,waste_time,stretch_roles,apply_with_most,required_less,remove_data,id,linkedin_member_id,email,verification_data,token_expires_at,last_synced_at,session_id,user_id,user_display_name,…,ip_hash,user_agent,device_summary,is_logged_in,login_state_changed,created_at,updated_at,sub,email_verified,name,locale,given_name,family_name,picture,session_survey_id,df_age,df_face_confidence,df_dominant_gender,df_dominant_race,df_prob_male,df_prob_female,df_prob_white,df_prob_black,df_prob_asian,df_prob_indian,df_prob_middle_eastern,df_prob_latino,df_status,df_error,gg_gender_label,eth_asian,eth_hispanic,eth_nh_black,eth_nh_white,eth_race_label,eth_processing_status,ethnicolr_status
list[struct[15]],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,list[null],str,str,str,str,str,…,str,str,str,bool,bool,str,str,str,bool,str,struct[2],str,str,str,str,i64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,f64,f64,f64,str,str,str
"[{2038,""1295f2d4cfb3d6ded8462211ed26e674a32cb0e96ff4e563658ab2e5df9a4e81"",6,""2026-04-21 16:48:47"",""https://jobconnectionsproject.org/jobs/"",""/jobs/"","""",""https://jobconnectionsproject.org/"",""128.187.116.32"",""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"",622,""Search for Jobs"",false,false,true}, {2039,""1295f2d4cfb3d6ded8462211ed26e674a32cb0e96ff4e563658ab2e5df9a4e81"",6,""2026-04-21 16:48:51"",""https://jobconnectionsproject.org/jobs/corporate-executive-pastry-chef-troy-mi-us/"",""/jobs/corporate-executive-pastry-chef-troy-mi-us/"","""",""https://jobconnectionsproject.org/jobs/"",""128.187.116.32"",""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"",3590,""Corporate Executive Pastry Chef &#8211; Troy, MI, US"",false,false,true}, … {2047,""1295f2d4cfb3d6ded8462211ed26e674a32cb0e96ff4e563658ab2e5df9a4e81"",6,""2026-04-21 17:26:12"",""https://jobconnectionsproject.org/"",""/"","""",""https://jobconnectionsproject.org/survey/"",""128.187.116.32"",""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"",67,""Home"",false,false,true}]","""128.187.116.32""","""2026-04-21 04:49:16""","""8680426331""","""0""","""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""https://jobconnectionsproject.…","""Unlikely""","""Equally Likely and Unlikely""","""""","""""","""may or may not""","""""","""""","""""","""Some college""","""Multiracial or Biracial""","""A gender identity not listed h…","""Employed (part-time)""","""""","""Apply now on company website""","""""","""3""","""""","""3""","""""","""""",1,"""AKLfR3WKvk""","""porter.olson11@gmail.com""",[],"""2026-06-08 03:54:09""","""2026-04-09 03:54:10""","""1295f2d4cfb3d6ded8462211ed26e6…","""6""","""Porter Olson""",…,"""a2479127ba3990c4c2859b95174800…","""Mozilla/5.0 (Windows NT 10.0; …","""Desktop / Chrome / Windows""",true,true,"""2026-04-21 16:48:47""","""2026-04-21 17:27:52""","""AKLfR3WKvk""",true,"""Porter Olson""","{""US"",""en""}","""Porter""","""Olson""","""https://media.licdn.com/dms/im…","""1295f2d4cfb3d6ded8462211ed26e6…",25,0.92,"""Man""","""white""",99.571747,0.42825,94.859116,0.000324,0.029293,0.011143,2.819904,2.280221,"""ok""",null,"""male""",0.000641,0.00363,0.007302,0.988427,"""nh_white""","""processed""","""ok"""
"[{2038,""1295f2d4cfb3d6ded8462211ed26e674a32cb0e96ff4e563658ab2e5df9a4e81"",6,""2026-04-21 16:48:47"",""https://jobconnectionsproject.org/jobs/"",""/jobs/"","""",""https://jobconnectionsproject.org/"",""128.187.116.32"",""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.